# 04 · Spatial-Orbital MP2

MP2 (Møller-Plesset perturbation theory to second order) 是 Hartree-Fock 之后最常见的相关能修正。

本 notebook 从 RHF canonical orbital 出发，实现最简单的闭壳层 spatial-orbital MP2：

1. 用 PySCF 做 RHF，拿到 MO 系数和轨道能量
2. 用 PySCF `ao2mo` 得到 s1 symmetry 的完整 MO 积分
3. 用 `einsum` 对照验证积分变换
4. 写出闭壳层 MP2 相关能公式
5. 用四重循环和向量化两种方式实现
6. 与 PySCF MP2 对比

这里的目标是让代码足够短，适合方法验证；真正的高性能 MP2 会继续利用积分对称性、分块和磁盘/内存管理。

---

## 1 · 从 RHF 到 MP2

Hartree-Fock 给出一个单行列式参考态 $|\text{HF}\rangle$，但电子之间的瞬时避让没有被完全描述。
MP2 把剩下的电子相关作为二阶微扰来估计。

对闭壳层 canonical RHF 轨道，MP2 相关能可以写成 spatial-orbital 形式：

$$
\boxed{
E_{\text{MP2}}^{(2)} = \sum_{ij}^{\text{occ}} \sum_{ab}^{\text{vir}}
\frac{(ia|jb)\left[2(ia|jb) - (ib|ja)\right]}
{\varepsilon_i + \varepsilon_j - \varepsilon_a - \varepsilon_b}
}
$$

其中：

| 指标 | 含义 |
|------|------|
| $i,j$ | 占据空间轨道，每个轨道有一对 alpha/beta 电子 |
| $a,b$ | 虚空间轨道 |
| $\varepsilon_p$ | RHF canonical orbital energy |
| $(pq|rs)$ | chemist notation 双电子积分 |

分母通常为负数，因为占据轨道能量低于虚轨道能量，因此 MP2 相关能通常为负。

---

## 2 · 分子设定与 RHF 参考态

为了让四维张量保持小尺寸，先用 `H2O/STO-3G`。如果想看更实际的数值，可以把 `basis='sto-3g'` 改成 `basis='cc-pvdz'`。

In [1]:
import numpy as np
from pyscf import gto, scf, mp, ao2mo

# -------- H2O / STO-3G --------
mol = gto.M(atom="O 0 0 0; H 0 -0.757 0.587; H 0 0.757 0.587", basis="sto-3g")

mf = scf.RHF(mol)
mf.verbose = 0
mf.kernel()

C = mf.mo_coeff                         # AO -> MO coefficient, shape (nao, nmo)
mo_energy = mf.mo_energy                # canonical RHF orbital energies
nmo = C.shape[1]
nelec = mol.nelectron
nocc = nelec // 2
nvir = nmo - nocc

print(f"RHF energy = {mf.e_tot:.10f} Hartree")
print(f"nmo = {nmo}, nelec = {nelec}, nocc = {nocc}, nvir = {nvir}")
print(f"occupied orbital energies: {mo_energy[:nocc]}")
print(f"virtual  orbital energies: {mo_energy[nocc:]}")

RHF energy = -74.9630631297 Hartree
nmo = 7, nelec = 10, nocc = 5, nvir = 2
occupied orbital energies: [-20.24196606  -1.26816062  -0.61738504  -0.45315324  -0.39127369]
virtual  orbital energies: [0.60513593 0.74124146]


---

## 3 · AO->MO 积分变换

FCI 一节已经手写过完整 AO->MO 四指标变换；在 MP2 这里，我们直接使用 PySCF 的 `ao2mo`。

`aosym="s1"` 表示不使用 AO 积分的置换对称性压缩，得到的数组可以直接 reshape 成完整的：

$$
(pq|rs), \quad \text{shape}=(n_{\text{mo}},n_{\text{mo}},n_{\text{mo}},n_{\text{mo}})
$$

下面仍然保留一个 `einsum` 对照：它非常适合验证公式和教学，但不一定是高性能计算里的最佳实现。

In [2]:
# eri_mo[p,q,r,s] = (pq|rs), chemist notation
eri_mo = ao2mo.kernel(mol, C, aosym="s1").reshape(nmo, nmo, nmo, nmo)

# Sanity check: the explicit einsum transform should give the same full tensor.
eri_ao = mol.intor("int2e_sph")
eri_mo_einsum = np.einsum("up,vq,kr,ls,uvkl->pqrs", C, C, C, C, eri_ao, optimize=True)
ao2mo_err = np.max(np.abs(eri_mo - eri_mo_einsum))

print(f"eri_mo shape = {eri_mo.shape}")
print(f"eri_mo size  = {eri_mo.size:,} elements")
print(f"max|ao2mo - einsum| = {ao2mo_err:.2e}")
assert ao2mo_err < 1e-10

eri_ao shape = (7, 7, 7, 7)
eri_mo shape = (7, 7, 7, 7)
eri_mo size  = 2,401 elements


---

## 4 · 四重循环实现

先用最直接的四重循环写一遍。这样最接近公式，也最容易检查每个指标的含义。

注意 chemist notation：

- `eri_mo[i,a,j,b]` = $(ia|jb)$
- `eri_mo[i,b,j,a]` = $(ib|ja)$

In [3]:
def mp2_energy_loop(mo_energy, eri_mo, nocc):
    nmo = len(mo_energy)
    emp2 = 0.0

    for i in range(nocc):
        for j in range(nocc):
            for a in range(nocc, nmo):
                for b in range(nocc, nmo):
                    iajb = eri_mo[i, a, j, b]      # (ia|jb), Coulomb-like term
                    ibja = eri_mo[i, b, j, a]      # (ib|ja), exchange-like term
                    denom = mo_energy[i] + mo_energy[j] - mo_energy[a] - mo_energy[b]
                    emp2 += iajb * (2.0 * iajb - ibja) / denom

    return emp2

emp2_loop = mp2_energy_loop(mo_energy, eri_mo, nocc)
print(f"E_MP2 correlation (loop) = {emp2_loop:.10f} Hartree")
print(f"E_RHF + E_MP2           = {mf.e_tot + emp2_loop:.10f} Hartree")

E_MP2 correlation (loop) = -0.0355668182 Hartree
E_RHF + E_MP2           = -74.9986299479 Hartree


---

## 5 · `einsum` 向量化实现

四重循环对教学很好，但 NumPy 更适合一次性操作整个张量。

先切出 MP2 真正需要的块：

$$
(ia|jb), \quad i,j \in \text{occ}, \; a,b \in \text{vir}
$$

数组 `iajb` 的形状是 `(nocc, nvir, nocc, nvir)`，四个轴依次对应 `i, a, j, b`。

In [4]:
def mp2_energy_einsum(mo_energy, eri_mo, nocc):
    eps_occ = mo_energy[:nocc]
    eps_vir = mo_energy[nocc:]

    # iajb[i,a,j,b] = (ia|jb)
    iajb = eri_mo[:nocc, nocc:, :nocc, nocc:]

    # ibja[i,a,j,b] = (ib|ja), obtained by swapping a <-> b inside iajb
    ibja = iajb.transpose(0, 3, 2, 1)

    denom = (
        eps_occ[:, None, None, None]
        - eps_vir[None, :, None, None]
        + eps_occ[None, None, :, None]
        - eps_vir[None, None, None, :]
    )

    emp2 = np.einsum("iajb,iajb,iajb->", iajb, 2.0 * iajb - ibja, 1.0 / denom, optimize=True)
    return emp2

emp2_einsum = mp2_energy_einsum(mo_energy, eri_mo, nocc)

print(f"E_MP2 correlation (einsum) = {emp2_einsum:.10f} Hartree")
print(f"loop - einsum difference   = {abs(emp2_loop - emp2_einsum):.2e} Hartree")

E_MP2 correlation (einsum) = -0.0355668182 Hartree
loop - einsum difference   = 0.00e+00 Hartree


---

## 6 · 分母与振幅

MP2 也常写成双激发振幅的形式：

$$
t_{ij}^{ab} = \frac{(ia|jb)}{\varepsilon_i + \varepsilon_j - \varepsilon_a - \varepsilon_b}
$$

然后能量为：

$$
E_{\text{MP2}}^{(2)} = \sum_{ijab} (ia|jb) \left[2t_{ij}^{ab} - t_{ij}^{ba}\right]
$$

这和后面的 CCSD 记号会非常接近。

In [5]:
eps_occ = mo_energy[:nocc]
eps_vir = mo_energy[nocc:]
iajb = eri_mo[:nocc, nocc:, :nocc, nocc:]

denom = (
    eps_occ[:, None, None, None]
    - eps_vir[None, :, None, None]
    + eps_occ[None, None, :, None]
    - eps_vir[None, None, None, :]
)

t2 = iajb / denom
emp2_from_t2 = np.einsum(
    "iajb,iajb->",
    iajb,
    2.0 * t2 - t2.transpose(0, 3, 2, 1),
    optimize=True,
)

print(f"t2 shape = {t2.shape}  # (i,a,j,b)")
print(f"denominator range = [{denom.min():.6f}, {denom.max():.6f}] Hartree")
print(f"E_MP2 from t2 = {emp2_from_t2:.10f} Hartree")

t2 shape = (5, 2, 5, 2)  # (i,a,j,b)
denominator range = [-41.966415, -1.992819] Hartree
E_MP2 from t2 = -0.0355668182 Hartree


---

## 7 · 与 PySCF MP2 对照

In [6]:
mp2_ref = mp.MP2(mf)
mp2_ref.verbose = 0
emp2_ref, t2_ref = mp2_ref.kernel()

print(f"Our MP2 correlation:   {emp2_einsum:.10f} Hartree")
print(f"PySCF MP2 correlation: {emp2_ref:.10f} Hartree")
print(f"Difference:            {abs(emp2_einsum - emp2_ref):.2e} Hartree")

print(f"\nOur MP2 total energy:   {mf.e_tot + emp2_einsum:.10f} Hartree")
print(f"PySCF MP2 total energy: {mp2_ref.e_tot:.10f} Hartree")

Our MP2 correlation:   -0.0355668182 Hartree
PySCF MP2 correlation: -0.0355668182 Hartree
Difference:            1.39e-17 Hartree

Our MP2 total energy:   -74.9986299479 Hartree
PySCF MP2 total energy: -74.9986299479 Hartree


---

## 8 · 总结

| 模块 | 核心内容 |
|------|----------|
| **参考态** | canonical RHF orbital energy 和 MO coefficient |
| **积分变换** | `ao2mo.kernel(mol, C, aosym="s1").reshape(nmo,nmo,nmo,nmo)` |
| **验证思路** | `einsum` 适合做公式检查和小体系验证，但不是高性能路线 |
| **占据/虚轨道** | `i,j` 是 occupied spatial orbitals，`a,b` 是 virtual spatial orbitals |
| **直接项** | $(ia|jb)$ |
| **交换项** | $(ib|ja)$ |
| **能量公式** | $\sum_{ijab}(ia|jb)[2(ia|jb)-(ib|ja)]/\Delta_{ij}^{ab}$ |
| **分母** | $\Delta_{ij}^{ab}=\varepsilon_i+\varepsilon_j-\varepsilon_a-\varepsilon_b$ |

Spatial-orbital MP2 已经把闭壳层自旋求和做掉了，所以公式里出现了 `2 * Coulomb - Exchange`。
下一份 notebook 会显式构造 spin orbital，在 spin-orbital 空间里用反对称化积分和 $1/4$ 因子得到同一个结果。